# Data cleaning

## Import required libraries

In [1]:
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

## Load the dataset

In [2]:
DATA_PATH = Path("../data/raw/train.csv")

df = pd.read_csv(DATA_PATH)

## Convert `Order Date` and `Ship Date` to pandas datetime format

In [3]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y')

## Sorting data by `Order Date`

In [4]:
df.sort_values(by=['Order Date'], inplace=True, ascending=True)

## Delete `Row ID` column

In [5]:
df.drop(columns=['Row ID'], inplace=True)

## Filling null values in `Postal Code`

In [6]:
df[df['Postal Code'].isnull()]

,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
5274,CA-2016-162887,2016-11-07,2016-11-09,Second Class,SV-20785,Stewart Visinsky,Consumer,United States,Burlington,Vermont,NaN,East,FUR-CH-10000595,Furniture,Chairs,Safco Contoured Stacking Chairs,715.20
9741,CA-2016-117086,2016-11-08,2016-11-12,Standard Class,QJ-19255,Quincy Jones,Corporate,United States,Burlington,Vermont,NaN,East,FUR-BO-10004834,Furniture,Bookcases,"Riverside Palais Royal Lawyers Bookcase, Royal...",4404.90
9146,US-2017-165505,2017-01-23,2017-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,TEC-AC-10002926,Technology,Accessories,Logitech Wireless Marathon Mouse M705,99.98
9148,US-2017-165505,2017-01-23,2017-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,OFF-ST-10001526,Office Supplies,Storage,Iceberg Mobile Mega Data/Printer Cart,1564.29
9147,US-2017-165505,2017-01-23,2017-01-27,Standard Class,CB-12535,Claudia Bergmann,Corporate,United States,Burlington,Vermont,NaN,East,OFF-AR-10003477,Office Supplies,Art,4009 Highlighters,8.04
8798,US-2017-150140,2017-04-06,2017-04-10,Standard Class,VM-21685,Valerie Mitchum,Home Office,United States,Burlington,Vermont,NaN,East,TEC-PH-10002555,Technology,Phones,Nortel Meridian M5316 Digital phone,1294.75
9388,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-AP-10000828,Office Supplies,Appliances,Avanti 4.4 Cu. Ft. Refrigerator,542.94
9389,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-EN-10001509,Office Supplies,Envelopes,Poly String Tie Envelopes,2.04
9387,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-PA-10001970,Office Supplies,Paper,Xerox 1881,12.28
9386,US-2018-127292,2018-01-19,2018-01-23,Standard Class,RM-19375,Raymond Messe,Consumer,United States,Burlington,Vermont,NaN,East,OFF-PA-10000157,Office Supplies,Paper,Xerox 191,79.92


All missing values were found in the Postal Code column and corresponded to orders from Burlington, Vermont. 
Postal Code for Burlington is 05401.

In [7]:
df['Postal Code'] = df['Postal Code'].fillna(5401)

## Check data types and missing values¶

In [8]:
df.info()

<class 'pandas.DataFrame'>
Index: 9800 entries, 7980 to 5091
Data columns (total 17 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Order ID       9800 non-null   str           
 1   Order Date     9800 non-null   datetime64[us]
 2   Ship Date      9800 non-null   datetime64[us]
 3   Ship Mode      9800 non-null   str           
 4   Customer ID    9800 non-null   str           
 5   Customer Name  9800 non-null   str           
 6   Segment        9800 non-null   str           
 7   Country        9800 non-null   str           
 8   City           9800 non-null   str           
 9   State          9800 non-null   str           
 10  Postal Code    9800 non-null   float64       
 11  Region         9800 non-null   str           
 12  Product ID     9800 non-null   str           
 13  Category       9800 non-null   str           
 14  Sub-Category   9800 non-null   str           
 15  Product Name   9800 non-null   str

## Detect potential outliers in the `Sales` column

To identify unusually high or low sales values, the Interquartile Range (IQR) method is applied.

The lower and upper bounds are calculated as:

- Lower bound = Q1 − 1.5 × IQR
- Upper bound = Q3 + 1.5 × IQR

Transactions outside these bounds are considered potential outliers and will be reviewed to determine whether they are data errors or legitimate high-value sales.

In [9]:
Q1 = df["Sales"].quantile(0.25)
Q3 = df["Sales"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

In [10]:
outliers = df[(df["Sales"] < lower) | (df["Sales"] > upper)]

print(f"Number of outliers: {len(outliers)}")
outliers = outliers.sort_values(by='Sales', ascending=False)
outliers.head()

Number of outliers: 1145


,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales
2697,CA-2015-145317,2015-03-18,2015-03-23,Standard Class,SM-20320,Sean Miller,Home Office,United States,Jacksonville,Florida,32216.0,South,TEC-MA-10002412,Technology,Machines,Cisco TelePresence System EX90 Videoconferenci...,22638.480
6826,CA-2017-118689,2017-10-02,2017-10-09,Standard Class,TC-20980,Tamara Chand,Corporate,United States,Lafayette,Indiana,47905.0,Central,TEC-CO-10004722,Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,17499.950
8153,CA-2018-140151,2018-03-23,2018-03-25,First Class,RB-19360,Raymond Buch,Consumer,United States,Seattle,Washington,98115.0,West,TEC-CO-10004722,Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,13999.960
2623,CA-2018-127180,2018-10-22,2018-10-24,First Class,TA-21385,Tom Ashbrook,Home Office,United States,New York City,New York,10024.0,East,TEC-CO-10004722,Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,11199.968
4190,CA-2018-166709,2018-11-17,2018-11-22,Standard Class,HL-15040,Hunter Lopez,Consumer,United States,Newark,Delaware,19711.0,East,TEC-CO-10004722,Technology,Copiers,Canon imageCLASS 2200 Advanced Copier,10499.970


### Observation

The IQR method identified 1145 potential outliers.

The detected outliers correspond to expensive technology products, such as enterprise copiers and video conferencing equipment. Since these transactions appear to be legitimate high-value sales and not data entry errors, they will be retained in the dataset for subsequent analysis.

## Save clean data

In [11]:
df.to_csv("../data/processed/clean_data.csv", index=False)

---
## Save clean data to PostgreSQL

In [12]:
load_dotenv(override=True)

engine = create_engine(
    f"postgresql+psycopg://"
    f"{os.getenv('DB_USER')}:"
    f"{os.getenv('DB_PASSWORD')}@"
    f"{os.getenv('DB_HOST')}:"
    f"{os.getenv('DB_PORT')}/"
    f"{os.getenv('DB_NAME')}"
)

In [15]:
sql_df = df.copy()

sql_df.columns = (
    sql_df.columns
          .str.lower()
          .str.replace(" ", "_")
          .str.replace("-", "_")
)

sql_df.to_sql("orders", engine, if_exists="replace", index=False)

-1

Column names are converted to `snake_case` to follow SQL naming conventions and improve PostgreSQL compatibility.